In [67]:
import pandas as pd

df = pd.read_csv("ai4i2020.csv")

print(df.head())
print(df.shape)

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0    0    0  
2    0  

In [68]:
print(df.columns)

Index(['UDI', 'Product ID', 'Type', 'Air temperature [K]',
       'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]',
       'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF',
       'RNF'],
      dtype='object')


In [69]:
df = df.drop([
    "UDI",
    "Product ID"
], axis=1)

In [70]:
df = pd.get_dummies(df, columns=["Type"])

In [71]:
#One hot encoding
df[['Type_H','Type_L','Type_M']] = (
df[['Type_H','Type_L','Type_M']]
.astype(int)
)

In [72]:
X = df.drop([
"Machine failure",
"TWF",
"HDF",
"PWF",
"OSF",
"RNF"
], axis=1)

y = df["Machine failure"]

In [73]:
print(X.columns)

Index(['Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_H',
       'Type_L', 'Type_M'],
      dtype='object')


In [74]:
X.columns = (
    X.columns
    .str.replace('[', '', regex=False)
    .str.replace(']', '', regex=False)
    .str.replace('<', '', regex=False)
    .str.replace('>', '', regex=False)
    .str.replace(' ', '_')
)

In [75]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [76]:
from xgboost import XGBClassifier

model = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [77]:
print(X_train.dtypes)

Air_temperature_K        float64
Process_temperature_K    float64
Rotational_speed_rpm       int64
Torque_Nm                float64
Tool_wear_min              int64
Type_H                     int64
Type_L                     int64
Type_M                     int64
dtype: object


In [78]:
print(df["Machine failure"].value_counts())

Machine failure
0    9661
1     339
Name: count, dtype: int64


In [79]:
#handling imbalance
from xgboost import XGBClassifier

model = XGBClassifier(
    scale_pos_weight=9661/339
)

In [80]:
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [81]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.73      0.75      0.74        68

    accuracy                           0.98      2000
   macro avg       0.86      0.87      0.86      2000
weighted avg       0.98      0.98      0.98      2000



In [82]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.73      0.75      0.74        68

    accuracy                           0.98      2000
   macro avg       0.86      0.87      0.86      2000
weighted avg       0.98      0.98      0.98      2000



In [83]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1913   19]
 [  17   51]]


In [84]:
from sklearn.metrics import roc_auc_score

y_prob = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

ROC-AUC: 0.9707861405431737


In [85]:
sample = X_test.iloc[[0]] #the first row of the test set regardless of whether it's healthy or failed.

prediction = model.predict(sample)
prob = model.predict_proba(sample)

print(sample)
print("Prediction:", prediction[0])
print("Failure Probability:", prob[0][1])
print("Actual:", y_test.iloc[0])

      Air_temperature_K  Process_temperature_K  Rotational_speed_rpm  \
2997              300.5                  309.8                  1345   

      Torque_Nm  Tool_wear_min  Type_H  Type_L  Type_M  
2997       62.7            153       0       1       0  
Prediction: 0
Failure Probability: 0.035198357
Actual: 0


In [86]:
#check whether the model detects failures
failure_rows = X_test[y_test == 1] #gives only the rows in the test set where the machine actually failed

print(failure_rows.head())

      Air_temperature_K  Process_temperature_K  Rotational_speed_rpm  \
4851              303.7                  312.1                  1363   
1391              298.9                  310.2                  2737   
4495              302.6                  310.4                  1359   
8582              297.5                  308.1                  1334   
9664              299.1                  310.2                  1317   

      Torque_Nm  Tool_wear_min  Type_H  Type_L  Type_M  
4851       51.8             90       0       1       0  
1391        8.8            142       0       1       0  
4495       57.2             67       0       1       0  
8582       72.0            151       0       0       1  
9664       54.8            231       0       1       0  


In [87]:
sample = failure_rows.iloc[[0]]

prediction = model.predict(sample)
prob = model.predict_proba(sample)

print("Prediction:", prediction[0])
print("Failure Probability:", prob[0][1])

Prediction: 0
Failure Probability: 0.17605618


In [88]:
# Healthy machine
healthy_sample = X_test[y_test == 0].iloc[[0]]

print("Healthy Machine")
print("Prob:",
      model.predict_proba(healthy_sample)[0][1])

# Failed machine
failed_sample = X_test[y_test == 1].iloc[[0]]

print("Failed Machine")
print("Prob:",
      model.predict_proba(failed_sample)[0][1])

Healthy Machine
Prob: 0.035198357
Failed Machine
Prob: 0.17605618


In [89]:
failure_probability = float(model.predict_proba(sample)[0][1])

if failure_probability > 0.8:
    risk = "CRITICAL"
elif failure_probability > 0.5:
    risk = "HIGH"
elif failure_probability > 0.2:
    risk = "MEDIUM"
else:
    risk = "LOW"

print(failure_probability)
print(risk)

0.1760561764240265
LOW


In [90]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

                 Feature  Importance
2   Rotational_speed_rpm    0.334610
3              Torque_Nm    0.244790
4          Tool_wear_min    0.226171
0      Air_temperature_K    0.054098
7                 Type_M    0.046894
6                 Type_L    0.039027
1  Process_temperature_K    0.038822
5                 Type_H    0.015588


In [91]:
sample = X_test.iloc[[0]]

prediction = model.predict(sample)[0]

failure_probability = model.predict_proba(sample)[0][1]

print("\nPrediction:", prediction)
print("Failure Probability:", failure_probability)


Prediction: 0
Failure Probability: 0.035198357


In [92]:
import joblib
joblib.dump(model, "factoryguard_failure_model.pkl")

print("\nModel saved successfully.")


Model saved successfully.
